# Unit 15 — Assembly Language Solutions

Worked answers for the short-answer exercises, and reference programs for the programming exercises.
Each programming solution is also saved as a runnable program in the `assets` folder (for example `assets/ex4.py`); run it with `python assets/ex4.py < input.txt`.
Each short-answer answer is checked by a check cell that runs the same assembly program through a small evaluator kept in `assets/verify/asm_eval.py`.

## Exercise 1

Trace `ACC` line by line. The `DC` lines set `X` = 7, `Y` = −4, `Z` = 5 and `R` = 0 before the program starts.

| Line | `ACC` | Memory change |
|------|-------|---------------|
| `LOAD X` | 7 | |
| `ADD Y` | 7 + (−4) = 3 | |
| `MULT Z` | 3 × 5 = 15 | |
| `SUB X` | 15 − 7 = 8 | |
| `STORE R` | 8 | `R` = 8 |

**Answer:** `8`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import asm_eval

program = """\
X     DC    7
Y     DC    -4
Z     DC    5
R     DC    0
      LOAD  X
      ADD   Y
      MULT  Z
      SUB   X
      STORE R
      END
"""

result = asm_eval.run(program)
assert str(result["memory"]["R"]) == "8"

## Exercise 2

`DIV` keeps the whole-number part, rounding toward zero.

| Line | `ACC` | Memory change |
|------|-------|---------------|
| `LOAD P` | −17 | |
| `DIV Q` | −17 ÷ 5 = −3.4, so −3 | |
| `MULT W` | −3 × 3 = −9 | |
| `ADD =4` | −9 + 4 = −5 | |
| `DIV =2` | −5 ÷ 2 = −2.5, so −2 | |
| `STORE ANS` | −2 | `ANS` = −2 |

Rounding down instead (Python's `//`) would give −4 and then −9, −5, −3: the wrong answer.

**Answer:** `-2`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import asm_eval

program = """\
P     DC    -17
Q     DC    5
W     DC    3
ANS   DC    0
      LOAD  P
      DIV   Q
      MULT  W
      ADD   =4
      DIV   =2
      STORE ANS
      END
"""

result = asm_eval.run(program)
assert str(result["memory"]["ANS"]) == "-2"

## Exercise 3

Start with `A` = 6 and `B` = 0.

| Line | `ACC` | Memory change |
|------|-------|---------------|
| `LOAD =10` | 10 | |
| `SUB A` | 10 − 6 = 4 | |
| `STORE B` | 4 | `B` = 4 |
| `MULT =-5` | 4 × (−5) = −20 | |
| `ADD B` | −20 + 4 = −16 | |
| `STORE A` | −16 | `A` = −16 |
| `LOAD A` | −16 | |
| `DIV B` | −16 ÷ 4 = −4 | |
| `STORE B` | −4 | `B` = −4 |
| `PRINT A` | | prints −16 |
| `PRINT B` | | prints −4 |

**Answer:** `-16 -4`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import asm_eval

program = """\
A     DC    6
B     DC    0
      LOAD  =10
      SUB   A
      STORE B
      MULT  =-5
      ADD   B
      STORE A
      LOAD  A
      DIV   B
      STORE B
      PRINT A
      PRINT B
      END
"""

result = asm_eval.run(program)
assert str(" ".join(str(v) for v in result["printed"])) == "-16 -4"

## Exercise 4

**Notice:** read `A` and `B`, then compute the two values line by line.
`ACC` goes `A`, `A` × `A`, `A` × `A` − `B`, then `DIV =3`; later `B`, `B` − `A`, then `DIV =4`.
Python's `//` rounds down, so `div` divides the sizes and puts the sign back: the quotient is negative exactly when one of the two numbers is negative.

In [ ]:
def div(x, y):
    q = abs(x) // abs(y)
    if x * y < 0:
        q = -q
    return q


a = int(input())
b = int(input())
c = div(a * a - b, 3)
print(c)
d = div(b - a, 4)
print(d)

## Exercise 5

`K` = 3 counts the passes: each pass reads one number, prints `R`, and lowers `K` by 1; the loop repeats while `K` is still positive.

| Pass | `N` | Branch taken | `ACC` stored in `R` | Prints | `K` after |
|------|-----|--------------|---------------------|--------|-----------|
| 1 | −7 | `BL NEG` | −7 × (−3) = 21 | 21 | 2 (`BG LOOP`) |
| 2 | 0 | `BE ZERO` | `LOAD =100`: 100 | 100 | 1 (`BG LOOP`) |
| 3 | 12 | none | 12 × 2 = 24 | 24 | 0 (falls through to `END`) |

**Answer:** `21 100 24`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import asm_eval

program = """\
K     DC    3
LOOP  READ  N
      LOAD  N
      BL    NEG
      BE    ZERO
      MULT  =2
      BU    OUT
NEG   MULT  =-3
      BU    OUT
ZERO  LOAD  =100
OUT   STORE R
      PRINT R
      LOAD  K
      SUB   =1
      STORE K
      BG    LOOP
      END
"""

result = asm_eval.run(program, [-7, 0, 12])
assert str(" ".join(str(v) for v in result["printed"])) == "21 100 24"

## Exercise 6

Each pass adds `I` × `I` to `S`, then adds 2 to `I`, and loops back while `I` − 9 is negative.

| Pass | `I` at `TOP` | `I` × `I` | `S` after | `I` after | `I` − 9 | Branch |
|------|--------------|-----------|-----------|-----------|---------|--------|
| 1 | 1 | 1 | 1 | 3 | −6 | `BL TOP` |
| 2 | 3 | 9 | 10 | 5 | −4 | `BL TOP` |
| 3 | 5 | 25 | 35 | 7 | −2 | `BL TOP` |
| 4 | 7 | 49 | 84 | 9 | 0 | falls through to `END` |

`S` = 1 + 9 + 25 + 49, the squares of the odd numbers below 9.

**Answer:** `84`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import asm_eval

program = """\
I     DC    1
S     DC    0
TOP   LOAD  I
      MULT  I
      ADD   S
      STORE S
      LOAD  I
      ADD   =2
      STORE I
      SUB   =9
      BL    TOP
      END
"""

result = asm_eval.run(program)
assert str(result["memory"]["S"]) == "84"

## Exercise 7

Each pass prints `N`, then stores `N` − 6 and loops while that is positive.

| Pass | Prints | `N` after `STORE` | Branch |
|------|--------|-------------------|--------|
| 1 | 20 | 14 | `BG LOOP` |
| 2 | 14 | 8 | `BG LOOP` |
| 3 | 8 | 2 | `BG LOOP` |
| 4 | 2 | −4 | falls through |

After the loop, `PRINT N` prints −4.

**Answer:** `20 14 8 2 -4`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import asm_eval

program = """\
N     DC    20
LOOP  PRINT N
      LOAD  N
      SUB   =6
      STORE N
      BG    LOOP
      PRINT N
      END
"""

result = asm_eval.run(program)
assert str(" ".join(str(v) for v in result["printed"])) == "20 14 8 2 -4"

## Exercise 8

The program computes `X` − `Y`.
If it is positive, `BG POS` jumps and `D` = `X` − `Y`.
Otherwise it loads `Y` and subtracts `X`, so `D` = `Y` − `X`, which is 0 or positive.
Either way `D` is the non-negative one of the two differences: |`X` − `Y`|.

- **A.** Not always: `X` = 3, `Y` = 5 prints 2, not 5.
- **B.** Always: this is |`X` − `Y`|.
- **C.** Not always: `X` = 3, `Y` = 5 prints 2, not −2.
- **D.** Always: the larger of `X` − `Y` and `Y` − `X` is the one that is not negative, which is |`X` − `Y`|.
- **E.** Not always: `X` = 3, `Y` = 5 prints 2, not 0.

**Answer:** `B, D`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import asm_eval

program = """\
      READ  X
      READ  Y
      LOAD  X
      SUB   Y
      BG    POS
      LOAD  Y
      SUB   X
POS   STORE D
      PRINT D
      END
"""

# Every pair from a grid over -1000..1000 (step 25) plus every pair from -10..10.
values = sorted(set(range(-1000, 1001, 25)) | set(range(-10, 11)))
options = {
    "A": lambda x, y, p: p == max(x, y),
    "B": lambda x, y, p: p == abs(x - y),
    "C": lambda x, y, p: p == x - y,
    "D": lambda x, y, p: p == max(x - y, y - x),
    "E": lambda x, y, p: x >= y or p == 0,
}
ok = {label: True for label in options}
for x in values:
    for y in values:
        printed = asm_eval.run(program, [x, y])["printed"]
        assert len(printed) == 1
        for label, fits in options.items():
            if not fits(x, y, printed[0]):
                ok[label] = False
chosen = [label for label in options if ok[label]]
assert str(", ".join(chosen) or "NONE") == "B, D"

## Exercise 9

The loop adds each number to `T` and counts it in `C`, until it reads 0. The 9 after the 0 is never read.

| Read `X` | `T` after | `C` after |
|----------|-----------|-----------|
| 8 | 8 | 1 |
| −15 | −7 | 2 |
| 3 | −4 | 3 |
| −7 | −11 | 4 |
| 0 | `BE DONE` | |

At `DONE`: `LOAD T` gives −11, and `DIV C` gives −11 ÷ 4 = −2.75, so −2 (toward zero, not −3). `A` = −2.
The program prints `C` and then `A`.

**Answer:** `4 -2`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import asm_eval

program = """\
T     DC    0
C     DC    0
LOOP  READ  X
      LOAD  X
      BE    DONE
      ADD   T
      STORE T
      LOAD  C
      ADD   =1
      STORE C
      BU    LOOP
DONE  LOAD  T
      DIV   C
      STORE A
      PRINT C
      PRINT A
      END
"""

result = asm_eval.run(program, [8, -15, 3, -7, 0, 9])
assert str(" ".join(str(v) for v in result["printed"])) == "4 -2"

## Exercise 10

**Notice:** each pass computes `Q` = `N` `DIV` 10, and `N` − `Q` × 10 (the last digit) is added to `S`; then `N` becomes `Q` and the loop repeats while `N` is positive.
The loop runs at least once, since the branch is at the end, so it becomes a `while` loop with a flag.
For a negative `N`, `DIV` rounds toward zero, so `N` − `Q` × 10 is the last digit with a minus sign (for −47 it is −7), and `Q` = −4 is not positive, so the loop stops after one pass.

In [ ]:
def div(x, y):
    q = abs(x) // abs(y)
    if x * y < 0:
        q = -q
    return q


s = 0
n = int(input())
going = True
while going:
    q = div(n, 10)
    s = s + n - q * 10
    n = q
    going = n > 0
print(s)

## Exercise 11

**Notice:** split each line; the opcode is the first word and the value is the second word without its `=`.
`ACC` starts at 0 and changes as each opcode says; `DIV` uses the same toward-zero `div` as Exercise 4.

In [ ]:
def div(x, y):
    q = abs(x) // abs(y)
    if x * y < 0:
        q = -q
    return q


n = int(input())
acc = 0
for k in range(n):
    parts = input().split()
    op = parts[0]
    loc = parts[1]
    v = int(loc[1:len(loc)])
    if op == "LOAD":
        acc = v
    elif op == "ADD":
        acc = acc + v
    elif op == "SUB":
        acc = acc - v
    elif op == "MULT":
        acc = acc * v
    elif op == "DIV":
        acc = div(acc, v)
    print(acc)

## Exercise 12

Each pass adds 1 to `C` and computes `A` − `B`.
If it is positive, `A` becomes `A` − `B`; if negative, `B` becomes `B` − `A`; if 0, the program stops.

| Pass | `C` | `A` | `B` | `A` − `B` | Change |
|------|-----|-----|-----|-----------|--------|
| 1 | 1 | 84 | 36 | 48 | `A` = 48 |
| 2 | 2 | 48 | 36 | 12 | `A` = 12 |
| 3 | 3 | 12 | 36 | −24 | `SWAP`: `B` = 36 − 12 = 24 |
| 4 | 4 | 12 | 24 | −12 | `SWAP`: `B` = 24 − 12 = 12 |
| 5 | 5 | 12 | 12 | 0 | `BE DONE` |

This is Euclid's subtraction method: `A` ends as the greatest common divisor of 84 and 36, which is 12.

**Answer:** `12 5`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import asm_eval

program = """\
A     DC    84
B     DC    36
C     DC    0
LOOP  LOAD  C
      ADD   =1
      STORE C
      LOAD  A
      SUB   B
      BE    DONE
      BL    SWAP
      STORE A
      BU    LOOP
SWAP  LOAD  B
      SUB   A
      STORE B
      BU    LOOP
DONE  PRINT A
      PRINT C
      END
"""

result = asm_eval.run(program)
assert str(" ".join(str(v) for v in result["printed"])) == "12 5"

## Exercise 13

`P` starts at 1.
Each pass lowers `N` by 1 and replaces `P` with `P` + `P`, doubling it, until `N` reaches 0.
So there are exactly `N` doublings, and the program prints 2 to the power `N`: 1, 2, 4, 8, 16, … (for `N` = 19 it is 524,288, still inside ±999,999).

| `N` | 0 | 1 | 2 | 3 |
|-----|---|---|---|---|
| prints | 1 | 2 | 4 | 8 |
| **A.** 2 × `N` | 0 | 2 | 4 | 6 |
| **C.** `N` × `N` | 0 | 1 | 4 | 9 |
| **E.** `N` + 1 | 1 | 2 | 3 | 4 |

- **A.** Fails at `N` = 0 (0, not 1).
- **B.** Always.
- **C.** Fails at `N` = 0 (0, not 1).
- **D.** Always: starting at 1 and doubling `N` times is 2 to the power `N`.
- **E.** Fails at `N` = 2 (3, not 4).

**Answer:** `B, D`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import asm_eval

program = """\
      READ  N
      LOAD  =1
      STORE P
LOOP  LOAD  N
      BE    DONE
      SUB   =1
      STORE N
      LOAD  P
      ADD   P
      STORE P
      BU    LOOP
DONE  PRINT P
      END
"""

options = {
    "A": lambda n: 2 * n,
    "B": lambda n: 2 ** n,
    "C": lambda n: n * n,
    "D": lambda n: 2 ** n,
    "E": lambda n: n + 1,
}
ok = {label: True for label in options}
for n in range(0, 20):
    printed = asm_eval.run(program, [n])["printed"]
    assert len(printed) == 1
    for label, f in options.items():
        if printed[0] != f(n):
            ok[label] = False
# D is stated in words; double 1 by hand N times as a separate check.
for n in range(0, 20):
    p = 1
    for k in range(n):
        p = p * 2
    assert asm_eval.run(program, [n])["printed"] == [p]
chosen = [label for label in options if ok[label]]
assert str(", ".join(chosen) or "NONE") == "B, D"

## Exercise 14

Each pass computes `Q` = `N` `DIV` 2, `T` = `Q` × 2 and `D` = `N` − `T` (the remainder, 0 or 1), prints `D`, and sets `N` = `Q`. It loops while `Q` is positive.

| Pass | `N` | `Q` | `T` | Prints `D` | Branch |
|------|-----|-----|-----|------------|--------|
| 1 | 22 | 11 | 22 | 0 | `BG LOOP` |
| 2 | 11 | 5 | 10 | 1 | `BG LOOP` |
| 3 | 5 | 2 | 4 | 1 | `BG LOOP` |
| 4 | 2 | 1 | 2 | 0 | `BG LOOP` |
| 5 | 1 | 0 | 0 | 1 | falls through |

These are the binary digits of 22 (`10110`), printed from the last digit to the first.

**Answer:** `0 1 1 0 1`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import asm_eval

program = """\
      READ  N
LOOP  LOAD  N
      DIV   =2
      STORE Q
      MULT  =2
      STORE T
      LOAD  N
      SUB   T
      STORE D
      PRINT D
      LOAD  Q
      STORE N
      BG    LOOP
      END
"""

result = asm_eval.run(program, [22])
assert str(" ".join(str(v) for v in result["printed"])) == "0 1 1 0 1"

## Exercise 15

The outer loop adds 1 to `I` and stops when `I` reaches 5, so it runs for `I` = 1, 2, 3, 4.
For each `I`, the inner loop starts `J` at `I` and adds `J` to `S`, lowering `J` by 1 while it stays positive: it adds `I` + (`I` − 1) + … + 1.

| `I` | Inner loop adds | `S` after |
|-----|-----------------|-----------|
| 1 | 1 | 1 |
| 2 | 2 + 1 = 3 | 4 |
| 3 | 3 + 2 + 1 = 6 | 10 |
| 4 | 4 + 3 + 2 + 1 = 10 | 20 |
| 5 | `BE DONE` | |

**Answer:** `20`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import asm_eval

program = """\
I     DC    0
S     DC    0
OUTER LOAD  I
      ADD   =1
      STORE I
      SUB   =5
      BE    DONE
      LOAD  I
      STORE J
INNER LOAD  S
      ADD   J
      STORE S
      LOAD  J
      SUB   =1
      STORE J
      BG    INNER
      BU    OUTER
DONE  PRINT S
      END
"""

result = asm_eval.run(program)
assert str(" ".join(str(v) for v in result["printed"])) == "20"

## Exercise 16

**Notice:** the loop stops when `N` − 1 is 0.
The lines `N DIV 2`, `MULT =2`, `SUB N` give 0 exactly when `N` is even; then `N` is halved, otherwise it becomes 3 × `N` + 1.
`C` counts the passes. Every `N` stays positive here, so Python's `//` and `%` give the same results as `DIV`.

In [ ]:
c = 0
n = int(input())
while n - 1 != 0:
    if n % 2 == 0:
        n = n // 2
    else:
        n = n * 3 + 1
    c = c + 1
print(c)

## Exercise 17

**Notice:** read the whole program first.
Each line becomes an opcode and a `LOC` (an empty string when there is none); a first word that is not an opcode is a label, and `where` remembers its line number.
`DC` lines fill `memory` before anything runs.
Then run from line 0: `line` moves to the next line unless a branch sets it to a label's line, and the loop stops at `END`.
`value` reads immediate data `=value` or a location, and `div` rounds toward zero.

In [ ]:
OPCODES = ["LOAD", "STORE", "ADD", "SUB", "MULT", "DIV", "BE", "BG", "BL", "BU",
           "READ", "PRINT", "DC", "END"]


def div(x, y):
    q = abs(x) // abs(y)
    if x * y < 0:
        q = -q
    return q


def value(loc, memory):
    if loc[0] == "=":
        return int(loc[1:len(loc)])
    return memory[loc]


n = int(input())
ops = []
locs = []
where = {}
memory = {}
for k in range(n):
    parts = input().split()
    first = 0
    if parts[0] not in OPCODES:
        where[parts[0]] = k
        first = 1
    op = parts[first]
    loc = ""
    if first + 1 < len(parts):
        loc = parts[first + 1]
    if op == "DC":
        memory[parts[0]] = int(loc)
    ops.append(op)
    locs.append(loc)
m = int(input())
inputs = []
for k in range(m):
    inputs.append(int(input()))

next_input = 0
acc = 0
line = 0
while ops[line] != "END":
    op = ops[line]
    loc = locs[line]
    line = line + 1
    if op == "LOAD":
        acc = value(loc, memory)
    elif op == "STORE":
        memory[loc] = acc
    elif op == "ADD":
        acc = acc + value(loc, memory)
    elif op == "SUB":
        acc = acc - value(loc, memory)
    elif op == "MULT":
        acc = acc * value(loc, memory)
    elif op == "DIV":
        acc = div(acc, value(loc, memory))
    elif op == "BE":
        if acc == 0:
            line = where[loc]
    elif op == "BG":
        if acc > 0:
            line = where[loc]
    elif op == "BL":
        if acc < 0:
            line = where[loc]
    elif op == "BU":
        line = where[loc]
    elif op == "READ":
        memory[loc] = inputs[next_input]
        next_input = next_input + 1
    elif op == "PRINT":
        print(memory[loc])

## Exercise 18

`N` goes 10, 11, …, 30; the program stops when `N` − `LIM` is positive (`N` = 31).
For each `N`, `D` starts at 2.
At `TRY`, if `D` × `D` − `N` is positive, no divisor up to the square root was found and `N` is printed.
Otherwise `N` `DIV` `D` × `D` − `N` is 0 exactly when `D` divides `N`; then `N` is not prime and the program goes to `NEXT`.
Otherwise `D` goes up by 1.

- `N` = 10: `D` = 2 divides it.
- `N` = 11: `D` = 2, 3 do not divide it; at `D` = 4, 16 − 11 > 0. Prints 11.
- `N` = 12, 14, 16, 18, 20, 22, 24, 26, 28, 30: `D` = 2 divides them.
- `N` = 15, 21, 27: `D` = 3 divides them.
- `N` = 25: at `D` = 5, 25 − 25 = 0 is not positive, and `D` = 5 divides 25.
- `N` = 13, 17, 19, 23, 29: no `D` divides them before `D` × `D` passes `N`. Printed.

The program prints the primes from 10 to 30.

**Answer:** `11 13 17 19 23 29`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import asm_eval

program = """\
N     DC    9
LIM   DC    30
NEXT  LOAD  N
      ADD   =1
      STORE N
      SUB   LIM
      BG    DONE
      LOAD  =2
      STORE D
TRY   LOAD  D
      MULT  D
      SUB   N
      BG    PRIME
      LOAD  N
      DIV   D
      MULT  D
      SUB   N
      BE    NEXT
      LOAD  D
      ADD   =1
      STORE D
      BU    TRY
PRIME PRINT N
      BU    NEXT
DONE  END
"""

result = asm_eval.run(program)
assert str(" ".join(str(v) for v in result["printed"])) == "11 13 17 19 23 29"